# RUSLE-based soil loss mapping for Madagascar with LS Factor 1 Variant   
  

This notebook computes **annual soil loss snapshots** for Madagascar  using the **Revised Universal Soil Loss Equation (RUSLE)**. The iplementation runs on Google Earth Engine (GEE) via the python Api. Interactive visualization was done with geemap in jupyter. Preprocessing of DEM (Digital Elevation model) and final mapping and visualization was done in Arcgis.


RUSLE (Revised Universal Soil loss Equation): (Renard et al., 1997; USDA NRCS, 2022; Wischmeier & Smith, 1978)


A = R * K *L * S * C * P


where:

- $A$: average annual soil loss (t·ha⁻¹·yr⁻¹)  
- $R$: rainfall erosivity factor  
- $K$: soil erodibility factor  
- $L$: slope length factor  
- $S$: slope steepness factor  
- $C$: cover–management factor (derived from Sentinel-2 NDVI)  
- $P$: support practice factor  


In [ ]:
!pip -q install --upgrade earthengine-api geemap folium


import ee
import geemap
import math
import os
import xarray
import numpy as np





   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.0/478.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.7 MB/s eta 0:00:00


In [ ]:
ee.Authenticate()
ee.Initialize(project='rusle-476614')

## 2. Study area, years, and projection

I define the study area using a boundary polygon extrcted from the LSIB simplified international boundaries dataset from the earth engine data catalog (USDOS/LSIB_SIMPLE/2017).

- **AOI** = Madagascar.  
- **Years** = mixed
- **Projection** = EPSG:32738'(Tananarive (Paris) / Laborde Grid approximation):



In [ ]:
#Study area (aoi) is defined

#Area of interest
countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
madagascar = countries.filter(ee.Filter.eq("country_na", "Madagascar")).geometry()
aoi = madagascar

#Target projection
target_crs = 'EPSG:32738'
target_scale = 30
nodata = -9999

#Digital elevation model as master grid
dem = ee.Image('projects/rusle-476614/assets/dem_merged_reproj_clipped_fill').clip(aoi)
dem30 = dem.reproject(crs=target_crs, scale=target_scale).toFloat()

target_proj = dem30.projection()

#inspect grid
print('Reference projection:', target_proj.getInfo())





Reference projection: {'type': 'Projection', 'crs': 'EPSG:32738', 'transform': [30, 0, 0, 0, -30, 0]}


## 2.1 Helper Function to harmonize all data inputs

- Analysis grid: 30m
- Downsampling with resample(): Bilinear, Nearest
- Reducing resolution: area-based aggregation (pixel becomes mean of all finer pixels that fall inside of it)




In [ ]:
def harmonize_to_30m(img, *,
                     kind: str,
                     coarse_method: str = "nearest",
                     fine_reducer=ee.Reducer.mean(),
                     max_pixels: int = 1024):

    if kind == "fine":
        return (
            img
            .setDefaultProjection(target_proj)   # critical
            .reduceResolution(reducer=fine_reducer, maxPixels=max_pixels)
            .reproject(target_proj)
            .toFloat()
        )

    elif kind == "coarse":
        if coarse_method in ("bilinear", "bicubic"):
            img = img.resample(coarse_method)
        elif coarse_method == "nearest":
            img = img.resample("nearest")
        else:
            raise ValueError("Invalid coarse_method")

        return (
            img
            .reproject(target_proj)
            .toFloat()
        )

    elif kind == "native30":
        return (
            img
            .reproject(target_proj)
            .toFloat()
        )

    else:
        raise ValueError("Invalid kind")

In [ ]:
# Get CRS string
target_crs = target_proj.crs().getInfo()

# Get affine transform as a 6-number list
target_transform = dem.projection().getInfo()['transform']

print(target_crs)
print(target_transform)

EPSG:32738
[30, 0, 316615.5059, 0, -30, 8675189.2741]


## 3. Rainfall erosivity factor (R)

The R factor represents the erosive power of rainfall (MJ·mm·ha⁻¹·h⁻¹·yr⁻¹). The dataset GloRESatE, a global mean annual rainfall erosivity product integrating satellite and reanalyiss with thousands of gauge locations, was used. In this application, it is loaded as a single year static raster.

Dataset: GloReSatE (Das et al., 2024).

Resolution: 0.1 deg, approx 11km

 Unit: MJ·mm·ha⁻¹·h⁻¹·yr⁻¹

 Temporal coverage: mean of analysis years (1998-2021)

Characteristics: derived product

Limitations for Madagascar
- no stations in madagascar, so it is mostly driven by satellite/reanalysis rather than local observations
- Conversion factor of 30-min erosivity derived fr europe, not africa.
- Event trresholds ma ynot match humid tropical rainfall regimes
- Satellite/reanalysis precipitation uncertainty propagates into erosivity
- Spatial reoslution is coarse and may smooth out rainfall gradients



Reference: Das, S., Jain, M.K., Gupta, V., McGehee, R.P., Yin, S., de Mello, C.R., Azari, M., Borrelli, P. and Panagos, P., 2024. GloRESatE: A dataset for global rainfall erosivity derived from multi-source data. Scientific Data, 11:926.


In [ ]:


#Import of dataset
R_ok = ee.Image("projects/rusle-476614/assets/GloRESatE").clip(aoi).rename("R")

#Resampling with harmonize function to 30m
R30 = harmonize_to_30m(R_ok, kind="coarse", coarse_method="bilinear")




#Visualization
Map_R = geemap.Map()
Map_R.centerObject(aoi, 6)

r_vis = {
    "min": 0,
    "max": 8000,
    "palette": ["0000ff", "00ffff", "ffff00", "ff0000"]
}

Map_R.addLayer(R_ok, r_vis, "R factor")


Map_R.add_colorbar(
    vis_params=r_vis,
    label="R Factor Value",
    layer_name="R factor"
)


Map_R.addLayerControl()
Map_R

Map(center=[-19.327868921217462, 46.73598435731322], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
#export t fee tooolbox
asset_id = "projects/rusle-476614/assets/Rfactor"

target_crs = target_proj.crs().getInfo()
target_transform = dem.projection().getInfo()['transform']   # or dem30.projection().getInfo()['transform']

task = ee.batch.Export.image.toAsset(
    image=R30.toFloat(),
    description="RUSLE_F1_export",
    assetId=asset_id,
    region=aoi,
    crs=target_crs,
    crsTransform=target_transform,
    maxPixels=1e13
)

task.start()

## 4. Soil erodibility factor (K)

The **K factor** describes how easily soil is detached and transported under standard conditions.  Global soil erodibility (RUSLE K-factor) map where the classic Wischmeier & Smith (1978) texture-based K equation is modified with measured saturated hydraulic conductivity (Ksat) to better capture soil hydraulic properties.

Dataset:K_Ksat

Resolution: Global coverage at 1 km (~30 arc-sec)

Unit: t·ha·h·ha⁻¹·MJ⁻¹·mm⁻¹

Temporal coverage: treated as static, soil samples form


Characteristics: soil property data (texture, organic carbon) with standard wischmeier and smith equation  and measurement of  saturated hydraulic conductivity ($K_{sat}$)

Limitations:



Gupta, S., Borrelli, P., Panagos, P., Alewell, C., 2024. An advanced global soil erodibility (K) assessment including the effects of saturated hydraulic conductivity. Science of The Total Environment 908, 168249. https://doi.org/10.1016/j.scitotenv.2023.168249

In [ ]:
#export t fee tooolbox


K = ee.Image("projects/rusle-476614/assets/K_Ksat").clip(aoi).rename("K")

#Resampling with harmonize function to 30m
K30   = harmonize_to_30m(K, kind="coarse", coarse_method="bilinear")

#Visualization
Map_K = geemap.Map()
Map_K.centerObject(aoi, 6)

k_vis = {
    "min": 0,
    "max": 0.03,
    "palette": ["ffffff", "ffcc99", "cc6400"]
}

Map_K.addLayer(K30, k_vis, "K factor")
Map_K.add_colorbar(
    vis_params=k_vis,
    label="K Factor Value (t·ha·h·ha⁻¹·MJ⁻¹·mm⁻¹)",
    layer_name="K factor"
)
Map_K.addLayerControl()

Map_K

Map(center=[-19.32786892121734, 46.73598435731314], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
asset_id = "projects/rusle-476614/assets/Kfactor"

target_crs = target_proj.crs().getInfo()
target_transform = dem.projection().getInfo()['transform']   # or dem30.projection().getInfo()['transform']

task = ee.batch.Export.image.toAsset(
    image=K30.toFloat(),
    description="RUSLE_F1_export",
    assetId=asset_id,
    region=aoi,
    crs=target_crs,
    crsTransform=target_transform,
    maxPixels=1e13
)

task.start()

## 5. LS Factor 1: L Factor by Desmet and Govers 1996 and  S Factor by McCool et al 1987


Desmet & Govers (1996). A GIS procedure for automatically calculating the USLE LS factor on topographically complex
landscape units. Journal of Soil and Water Conservation, 51 (5), 427–433

McCool et al., (1987). Revised slope steepness factor for the universal soil loss equation. Transactions of the ASAE, 30 (5), 1387–
1396. https://doi.org/10.13031/2013.30576



In [ ]:


#Import of dataset and Asset IDs

#Preprocessed Digital Elevation model
dem = ee.Image('projects/rusle-476614/assets/dem_merged_reproj_clipped_fill').clip(aoi).rename('elevation')
#Specific Catchment Area derived from DEM
sca = ee.image('projects/rusle-476614/assets/sca').clip(aoi)
#cellsize of image
sca30  = harmonize_to_30m(sca)
d = ee.Image.constant(30)

#DEM derivations: select "slope" and "aspect"
terrain = ee.Terrain.products(dem)
slope_deg = terrain.select('slope')
aspect_deg = terrain.select('aspect')

#Convert slope and aspect in radians
deg2rad = ee.Number(3.141592653589793).divide(180.0)
slope_rad = slope_deg.multiply(deg2rad)
aspect_rad = aspect_deg.multiply(deg2rad)

#Avoid zero slope numerical issues
eps = ee.Image.constant(1e-12)
slope_rad = slope_rad.max(eps)

sin_slope = slope_rad.sin()

# Aspect correction term for Desmet–Govers L
x = aspect_rad.sin().abs().add(aspect_rad.cos().abs())

# m exponent (McCool et al. 1989, as used in Desmet & Govers 1996)
# beta = (sin(slope)/0.0896) / (3*sin(slope)^0.8 + 0.56)
# m    = beta / (1 + beta)
beta = (
    sin_slope.divide(0.0896)
    .divide(sin_slope.pow(0.8).multiply(3.0).add(0.56))
)
m = beta.divide(beta.add(1.0))

# ---------------------------------------------------------------
# L factor — Desmet & Govers (1996), standard (unconstrained)
# L = [ (SCA + d^2)^(m+1) - SCA^(m+1) ] /
#     [ d^(m+2) * x^m * 22.13^m ]
# ---------------------------------------------------------------
L_num = (
    sca30.add(d.multiply(d))
    .pow(m.add(1.0))
    .subtract(sca30.pow(m.add(1.0)))
)

L_den = (
    d.pow(m.add(2.0))
    .multiply(x.pow(m))
    .multiply(ee.Image.constant(22.13).pow(m))
)

L = L_num.divide(L_den).rename('L')


#S factor BY McCool et al. (1987)
# threshold: sin(slope) corresponding to 9% slope (~5.143°)
# sin(arctan(0.09)) ≈ 0.08975817419
#
# S = 10.8 * sin(slope) + 0.03    for slope <  9%   (tan θ < 0.09)
# S = 16.8 * sin(slope) - 0.50    for slope >= 9%   (tan θ >= 0.09)
# ---------------------------------------------------------------
threshold = 0.08975817419

S_low  = sin_slope.multiply(10.8).add(0.03)
S_high = sin_slope.multiply(16.8).subtract(0.50)

# Combine the two branches using the slope threshold as a mask
mask_low = sin_slope.lt(threshold)
S = S_low.where(mask_low.Not(), S_high).rename('S')

# Combined LS (F1)
LS_F1 = L.multiply(S).rename('LS_F1').clip(aoi)



Map = geemap.Map()
Map.centerObject(LS_F2, 10)

Map.addLayer(L_cap, {'min': 0, 'max': 10}, 'L factor')
Map.addLayer(S, {'min': 0, 'max': 5}, 'S factor')
Map.addLayer(LS_F2, {'min': 0, 'max': 20}, 'LS factor')

Map


NameError: name 'S_high' is not defined

In [ ]:
#Export of final image asset LS Factorc "LS_F1" to then ingest to Arcgis via "gee-toolbox" for mapping and analysis
asset_id = "projects/rusle-476614/assets/LS_F1_clip"
task = ee.batch.Export.image.toAsset(
    image=LS_F1,
    region=aoi,
    scale=target_scale,
    crs='EPSG:32738',
    maxPixels=1e13,
    assetId=asset_id

)
task.start()

## 6. C factor: cover management

The **C factor** quantifies the effect of vegetation cover and management on erosion. We derive C from **Sentinel-2 NDVI**:
 Use `COPERNICUS/S2_SR_HARMONIZED` (Sentinel-2 MSI L2A).  
Mask clouds/shadows using the scene clasification layer (SCL).  
For each year:
   - making a median composite over the calendar year.  
   - compute the NDVI = (B8 − B4) / (B8 + B4).  
Converison of  to C using durigon et al linear relation:  
  
   C = C=(1-NDVI)\/2


In [ ]:
#Import of data
s2_sr = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

#Function to mask out clouds
def mask_s2_clouds(image):
    """Mask clouds and cloud shadows using the SCL band."""
    scl = image.select("SCL")

    good = (scl.neq(3)   # cloud shadow
            .And(scl.neq(8))   # medium probability cloud
            .And(scl.neq(9))   # high probability cloud
            .And(scl.neq(10))  # thin cirrus
            .And(scl.neq(11))) # snow/ice

    return image.updateMask(good)

#Set Analysis year for NDVI
start = ee.Date.fromYMD(2021, 1, 1)
end   = ee.Date.fromYMD(2022, 1, 1)

s2_2021 = (s2_sr
           .filterDate(start, end)
           .filterBounds(aoi)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
           .map(mask_s2_clouds))

composite = s2_2021.median().clip(aoi)

ndvi_2021 = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")

#Durigon et al Equation to derive C Factor

#Set parameters
alpha = 2
beta = 1

eps = 1e-6
denominator = ee.Image(beta).subtract(ndvi_2021).max(eps)

# C Factor:  C = C=(1-NDVI)\/2
C = ndvi_2021.multiply(-1).add(1).divide(2).multiply(0.1).rename('C')*0.1

C30   = harmonize_to_30m(C, kind="fine", fine_reducer=ee.Reducer.mean())






In [ ]:
#export of c factor
asset_id = "projects/rusle-476614/assets/C30_Cfactor_export"

target_crs = target_proj.crs().getInfo()
target_transform = dem.projection().getInfo()['transform']   # or dem30.projection().getInfo()['transform']

task = ee.batch.Export.image.toAsset(
    image=C30.toFloat(),
    description="C30_export",
    assetId=asset_id,
    region=aoi,
    crs=target_crs,
    crsTransform=target_transform,
    maxPixels=1e13
)

task.start()

## 8. Annual RUSLE soil loss computation

For each year $y$ we compute:

\[
A_y = R \cdot K \cdot L \cdot S \cdot C_y \cdot P
\]

- $R, K, L, S, P$ are static.  
- $C_y$ is the Sentinel-2-based C-factor for year y.  

With standard RUSLE units, $A_y$ is in **t·ha⁻¹·yr⁻¹**. We do **not** multiply by pixel area here.


In [ ]:
#RUSLE equation with annual soil loss

A_rusle_F1 = (
    R30.unmask(1)
    .multiply(K30.unmask(1))
    .multiply(LS.unmask(1))
    .multiply(C30.unmask(1))
    .rename("soil_loss")
)






In [ ]:
# Get CRS string
target_crs = target_proj.crs().getInfo()

# Get affine transform as a 6-number list
target_transform = dem.projection().getInfo()['transform']

print(target_crs)
print(target_transform)

EPSG:32738
[30, 0, 316615.5059, 0, -30, 8675189.2741]


In [ ]:


asset_id = "projects/rusle-476614/assets/RUSLE_F1"

target_crs = target_proj.crs().getInfo()
target_transform = dem.projection().getInfo()['transform']   # or dem30.projection().getInfo()['transform']

task = ee.batch.Export.image.toAsset(
    image=A_rusle_F1.toFloat(),
    description="RUSLE_F1_export",
    assetId=asset_id,
    region=aoi,
    crs=target_crs,
    crsTransform=target_transform,
    maxPixels=1e13
)

task.start()